# 🧪 06 - Describing Data Lab: Palmer Penguins

A research team has measured 344 penguins on three islands in Antarctica and sent you `penguins.csv`. They want a **numeric description** of the data before anyone draws a single chart:

- What's in the file, and what does each column mean?
- What's typical for each measurement, and how much does it vary?
- Are there outliers, and are they mistakes?
- How do the three species differ?
- Which measurements are related, and does that hold up within each species?

You'll answer with everything from notebooks 01–05, and save a small report. In **Module 08's lab**, you'll turn these same findings into figures, so keep your answers. You'll check them against the charts.

## ✅ Learning Goals
- Run a full descriptive workflow, from raw file to saved summary tables
- Choose statistics that match each column's type and shape
- Separate real outliers from errors, overall and within groups
- Compare groups fairly, and test whether a relationship survives splitting by group

```text
Load → Inspect → Clean (Module 06) → Data dictionary (01)
     → Center & spread (02) → Shape & outliers (03)
     → Compare groups (04) → Relationships (05) → Export the report
```

## 0. Setup
Everything this lab saves goes into **`06-OutputFiles`**. Upload that folder along with this notebook.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('06-OutputFiles')
OUT.mkdir(exist_ok=True)

MEASUREMENTS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

## 1. Load and Look (Change Nothing Yet)

In [2]:
penguins = pd.read_csv('penguins.csv')
print(penguins.shape)
penguins.head()

(344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [3]:
penguins.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


In [4]:
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

**One row = one penguin.** Four measurements are each missing twice, and `sex` is missing 11 times. Are the measurement gaps in the same two rows?

In [5]:
penguins[penguins[MEASUREMENTS].isna().any(axis=1)]

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN


Yes: two penguins with **no measurements at all**. They can't contribute anything to a description of measurements.

## 2. Clean (Module 06)
1. Drop the two rows with no measurements.
2. Keep penguins with unknown `sex`: their measurements are still useful. Just remember they'll drop out of any by-sex comparison.
3. Tidy `sex` from `MALE`/`FEMALE` to `Male`/`Female`.
4. Record meaning with dtypes (notebook 01): `species`, `island`, and `sex` are nominal → `category`.

In [6]:
clean = penguins.dropna(subset=MEASUREMENTS).copy()
clean['sex'] = clean['sex'].str.title()

for col in ['species', 'island', 'sex']:
    clean[col] = clean[col].astype('category')

print('Rows before:', len(penguins), '| after:', len(clean))
print(clean['sex'].value_counts(dropna=False))
clean.dtypes

Rows before: 344 | after: 342
sex
Male      168
Female    165
NaN         9
Name: count, dtype: int64


species              category
island               category
bill_length_mm        float64
bill_depth_mm         float64
flipper_length_mm     float64
body_mass_g           float64
sex                  category
dtype: object

### Cleaning check
The two rows with any missing measurement are also the two rows with **all** measurements missing in this file. Thus the instructor's `dropna(subset=MEASUREMENTS)` removes exactly the intended two rows here. No measurement was imputed, no complete measured penguin was discarded, and the nine remaining unknown-sex penguins stay available for measurement summaries.

## 3. Data Dictionary (Notebook 01)

In [7]:
data_dictionary = pd.DataFrame({
    'column': clean.columns,
    'meaning': ['Penguin species', 'Island where measured', 'Bill (beak) length',
                'Bill (beak) depth', 'Flipper length', 'Body mass', 'Sex'],
    'kind': ['qualitative', 'qualitative', 'quantitative', 'quantitative',
             'quantitative', 'quantitative', 'qualitative'],
    'scale': ['nominal', 'nominal', 'ratio', 'ratio', 'ratio', 'ratio', 'nominal'],
    'dtype': clean.dtypes.astype(str).values,
    'missing': clean.isna().sum().values,
    'distinct': clean.nunique().values,
})
data_dictionary.to_csv(OUT / 'penguins_data_dictionary.csv', index=False)
data_dictionary

,column,meaning,kind,scale,dtype,missing,distinct
0,species,Penguin species,qualitative,nominal,category,0,3
1,island,Island where measured,qualitative,nominal,category,0,3
2,bill_length_mm,Bill (beak) length,quantitative,ratio,float64,0,164
3,bill_depth_mm,Bill (beak) depth,quantitative,ratio,float64,0,80
4,flipper_length_mm,Flipper length,quantitative,ratio,float64,0,55
5,body_mass_g,Body mass,quantitative,ratio,float64,0,94
6,sex,Sex,qualitative,nominal,category,9,2


## 4. Center and Spread (Notebook 02)
One table with every statistic from notebooks 02 and 03, one row per measurement:

In [8]:
overall = clean[MEASUREMENTS].describe().T
overall['IQR'] = overall['75%'] - overall['25%']
overall['skew'] = clean[MEASUREMENTS].skew()
overall.round(2)

,count,mean,std,min,25%,50%,75%,max,IQR,skew
bill_length_mm,342.0,43.92,5.46,32.1,39.22,44.45,48.5,59.6,9.27,0.05
bill_depth_mm,342.0,17.15,1.97,13.1,15.60,17.30,18.7,21.5,3.10,-0.14
flipper_length_mm,342.0,200.92,14.06,172.0,190.00,197.00,213.0,231.0,23.00,0.35
body_mass_g,342.0,4201.75,801.95,2700.0,3550.00,4050.00,4750.0,6300.0,1200.00,0.47


All four measurements have a mean close to their median, and |skew| below 0.5: no strong lopsidedness like Titanic's fares. Mean and std are reasonable summaries here.

But "roughly symmetric overall" can hide something. This file mixes **three species** of very different sizes. Keep that in mind for section 6.

## 5. Shape and Outliers (Notebook 03)
Reuse the fence function from notebook 03, and count outliers in each measurement:

In [9]:
def iqr_fences(series, k=1.5):
    """Return the (lower, upper) outlier fences for a numeric Series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


for col in MEASUREMENTS:
    low, high = iqr_fences(clean[col])
    n_out = ((clean[col] < low) | (clean[col] > high)).sum()
    print(f'{col:<18} fences {low:8.1f} to {high:8.1f}   outliers: {n_out}')

bill_length_mm     fences     25.3 to     62.4   outliers: 0
bill_depth_mm      fences     10.9 to     23.3   outliers: 0
flipper_length_mm  fences    155.5 to    247.5   outliers: 0
body_mass_g        fences   1750.0 to   6550.0   outliers: 0


No outliers at all across the whole dataset. Now ask notebook 03's Practice 3 question: is a value unusual **for its own species**? Compute the fences within each species with `groupby().transform()`, which lines each group's statistic up with the original rows:

In [10]:
def outliers_within(df, col, group):
    """Boolean mask: is each value outside its own group's IQR fences?"""
    q1 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.25))
    q3 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    return (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)


within = pd.DataFrame({col: outliers_within(clean, col, 'species') for col in MEASUREMENTS})
within.sum()

bill_length_mm       1
bill_depth_mm        1
flipper_length_mm    2
body_mass_g          2
dtype: int64

In [11]:
clean[within.any(axis=1)].sort_values('species')

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
19,Adelie,Torgersen,46.0,21.5,194.0,4200.0,Male
28,Adelie,Biscoe,37.9,18.6,172.0,3150.0,Female
129,Adelie,Torgersen,44.1,18.0,210.0,4000.0,Male
189,Chinstrap,Dream,52.0,20.7,210.0,4800.0,Male
190,Chinstrap,Dream,46.9,16.6,192.0,2700.0,Female
253,Gentoo,Biscoe,59.6,17.0,230.0,6050.0,Male


A few penguins are unusual **for their species**, even though nothing is unusual overall. Look at the rows: are any values impossible? (A small Chinstrap is still a possible penguin. A 30 g penguin, or a 3-meter flipper, would not be.) These look like real, unusual animals, so we **keep** them.

> 🧠 Whether a value is an outlier depends on **which group you compare it to**.

## 6. Comparing Species (Notebook 04)

In [12]:
species_summary = (
    clean
    .groupby('species', observed=True)
    .agg(
        penguins=('body_mass_g', 'size'),
        mass_mean=('body_mass_g', 'mean'),
        mass_median=('body_mass_g', 'median'),
        mass_std=('body_mass_g', 'std'),
        flipper_mean=('flipper_length_mm', 'mean'),
        bill_length_mean=('bill_length_mm', 'mean'),
        bill_depth_mean=('bill_depth_mm', 'mean'),
    )
    .round(1)
)
species_summary.to_csv(OUT / 'species_summary.csv')
species_summary

,penguins,mass_mean,mass_median,mass_std,flipper_mean,bill_length_mean,bill_depth_mean
species,,,,,,,
Adelie,151,3700.7,3700.0,458.6,190.0,38.8,18.3
Chinstrap,68,3733.1,3700.0,384.3,195.8,48.8,18.4
Gentoo,123,5076.0,5000.0,504.1,217.2,47.5,15.0


Gentoo penguins are much heavier, with longer flippers, but **shallower** bills. Hold on to that "but". It matters in section 7.

### Where Each Species Lives

In [13]:
pd.crosstab(clean['species'], clean['island'], margins=True)

island,Biscoe,Dream,Torgersen,All
species,,,,
Adelie,44,56,51,151
Chinstrap,0,68,0,68
Gentoo,123,0,0,123
All,167,124,51,342


Chinstraps were measured only on Dream, and Gentoos only on Biscoe. So "Biscoe penguins are heavy" would really be a statement about **Gentoos**: island and species are tangled together (a confounder, notebook 04). Compare islands only **within** a species that lives on more than one (Adelie).

### Species × Sex

In [14]:
mass_by_species_sex = clean.pivot_table(
    values='body_mass_g', index='species', columns='sex', aggfunc='mean', observed=True,
).round(0)
mass_by_species_sex['difference'] = mass_by_species_sex['Male'] - mass_by_species_sex['Female']
mass_by_species_sex

sex,Female,Male,difference
species,,,
Adelie,3369.0,4043.0,674.0
Chinstrap,3527.0,3939.0,412.0
Gentoo,4680.0,5485.0,805.0


Within every species, males are heavier. The penguins with unknown sex aren't in this table (section 2).

## 7. Relationships (Notebook 05)

In [15]:
corr = clean[MEASUREMENTS].corr()
corr.round(2)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
bill_length_mm,1.00,-0.24,0.66,0.60
bill_depth_mm,-0.24,1.00,-0.58,-0.47
flipper_length_mm,0.66,-0.58,1.00,0.87
body_mass_g,0.60,-0.47,0.87,1.00


In [16]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
upper.stack().dropna().rename('r').sort_values(key=abs, ascending=False).round(3)

flipper_length_mm  body_mass_g          0.871
bill_length_mm     flipper_length_mm    0.656
                   body_mass_g          0.595
bill_depth_mm      flipper_length_mm   -0.584
                   body_mass_g         -0.472
bill_length_mm     bill_depth_mm       -0.235
Name: r, dtype: float64

Flipper length and body mass are very strongly related (bigger penguins, bigger flippers). Now look at **bill length vs. bill depth**: a weak **negative** correlation. Longer bills are shallower?

Check whether that holds up **within each species**:

In [17]:
rows = []
for species, group in clean.groupby('species', observed=True):
    rows.append({
        'species': species,
        'n': len(group),
        'r_bill_length_depth': group['bill_length_mm'].corr(group['bill_depth_mm']),
    })
rows.append({'species': 'ALL (combined)', 'n': len(clean),
             'r_bill_length_depth': clean['bill_length_mm'].corr(clean['bill_depth_mm'])})

bill_corr = pd.DataFrame(rows).round(3)
bill_corr.to_csv(OUT / 'bill_correlation_by_species.csv', index=False)
bill_corr

,species,n,r_bill_length_depth
0,Adelie,151,0.391
1,Chinstrap,68,0.654
2,Gentoo,123,0.643
3,ALL (combined),342,-0.235


🤯 **Within every species, the correlation is positive, and moderate to strong.** Combined, it's negative.

This is **Simpson's paradox**: a relationship that reverses when you combine groups. The species summary in section 6 explains how. Gentoos have long bills **and** shallow bills compared with the others. Pooling a "long and shallow" group with "shorter and deeper" groups creates a downward trend that doesn't exist inside any one species. `species` is the confounder.

You found this with numbers alone, but you had to know to split by species. In Module 08's lab, one scatter plot colored by species shows it instantly, with no need to know ahead of time.

## 8. Export the Report

In [18]:
clean.to_csv(OUT / 'cleaned_penguins.csv', index=False)
overall.round(2).to_csv(OUT / 'measurement_summary.csv')
corr.round(3).to_csv(OUT / 'correlation_matrix.csv')

print('Saved:', sorted(p.name for p in OUT.glob('*.csv')))

Saved: ['bill_correlation_by_species.csv', 'cleaned_penguins.csv', 'correlation_matrix.csv', 'measurement_summary.csv', 'penguins_data_dictionary.csv', 'sex_differences_by_species.csv', 'species_summary.csv']


## ❓ Lab Questions
Answer each in the Markdown cell below, with numbers from your output:
1. What does one row represent? How many rows did you remove, and why? Why did you **keep** penguins with unknown sex?
2. Which measurement varies the most **relative to its mean**? (Hint: divide `std` by `mean`.)
3. Why were there no outliers overall, but some within species? Did you keep or remove them, and why?
4. Which species is heaviest, and by how much (mean and median) compared with the lightest?
5. Why can't you compare islands directly? Which comparison would be fair?
6. Explain the bill length vs. bill depth result in your own words, as if to someone who has never heard of Simpson's paradox.

*Your answers here.*

**Responses:**

1. One row is one measured penguin. The raw file has 344 rows; two rows have all four measurements missing, so 342 remain. Nine retained penguins have unknown sex; their observed measurements still inform overall and species summaries, but they cannot enter male/female comparisons. The raw file had 11 missing sex entries, including the two removed rows.
2. Body mass varies most relative to its mean: std/mean ≈ 0.1909 (19.09%), compared with bill length 12.43%, bill depth 11.51%, and flipper length 7.00%. These ratios compare relative variation across different measurement units.
3. The pooled data mix species with different sizes, widening the overall IQR fences enough that none of the four measurements is flagged. Within species there are six flags across six penguins: one bill length, one bill depth, two flipper lengths, and two body masses. The observations are unusual relative to their own species but not obviously impossible; they are retained. A flag alone cannot verify an error or authenticate a measurement, so source records would be needed before correcting them.
4. Gentoo is heaviest (mean 5,076.02 g, median 5,000 g), while Adelie has the lowest mean (3,700.66 g; median 3,700 g). The mean difference is 1,375.35 g, and the median difference is 1,300 g. Chinstrap also has a 3,700 g median, so the lowest median is tied.
5. Species and island are entangled: all 68 Chinstraps are on Dream and all 123 Gentoos on Biscoe. Comparing pooled island means mixes species differences into the island comparison. A fairer descriptive comparison restricts to Adelie, which is observed on Biscoe (44), Dream (56), and Torgersen (51); sex and other sampling differences may still matter.
6. Overall, longer bills seem shallower (r ≈ −0.235), but within Adelie, Chinstrap, and Gentoo, length and depth increase together (r ≈ 0.391, 0.654, and 0.643). Gentoo penguins have longer but shallower bills than the other species. Mixing those different species creates a downward trend between groups that reverses their upward trends within groups. This reversal is Simpson's paradox; it does not show that making a bill longer causes its depth to change.


## 🏋️ Practice

### Practice 1 — Species Spread (easy)
For each species, compute the IQR of `flipper_length_mm`. Which species is the most variable, and is it also the one with the largest std?

In [19]:
# Your code here

flipper_spread = clean.groupby('species', observed=True)['flipper_length_mm'].agg(
    n='size', Q1=lambda s: s.quantile(.25), Q3=lambda s: s.quantile(.75), std='std')
flipper_spread['IQR'] = flipper_spread['Q3'] - flipper_spread['Q1']
print(flipper_spread.to_string())


             n     Q1     Q3       std   IQR
species                                     
Adelie     151  186.0  195.0  6.539457   9.0
Chinstrap   68  191.0  201.0  7.131894  10.0
Gentoo     123  212.0  221.0  6.484976   9.0


**Practice 1 response:** Chinstrap flipper length has the largest IQR, 10 mm, versus 9 mm for Adelie and Gentoo. It also has the largest sample standard deviation: 7.132 mm, versus 6.539 mm and 6.485 mm respectively. Both measures identify Chinstrap as the most variable here.

### Practice 2 — A Fair Island Comparison (medium)
Filter to **Adelie** penguins only (the one species on all three islands). Compare their body mass across islands with size, mean, median, and std. Is there a real island difference among Adelies?

In [20]:
# Your code here

adelie = clean.loc[clean['species'] == 'Adelie']
adelie_island_mass = adelie.groupby('island', observed=True)['body_mass_g'].agg(
    size='size', mean='mean', median='median', std='std')
print(adelie_island_mass.round(3).to_string())
print('Range of island means:', adelie_island_mass['mean'].max() - adelie_island_mass['mean'].min())


           size      mean  median      std
island                                    
Biscoe       44  3709.659  3750.0  487.734
Dream        56  3688.393  3575.0  455.146
Torgersen    51  3706.373  3700.0  445.108
Range of island means: 21.266233766233654


**Practice 2 response:** The Adelie island means are close: Biscoe 3,709.66 g, Dream 3,688.39 g, and Torgersen 3,706.37 g, a range of only 21.27 g. Their within-island sample standard deviations are about 445–488 g; median values are 3,750, 3,575, and 3,700 g. There are observed descriptive differences, but these summaries alone do not establish a population island effect or causation. The much larger pooled-island differences are strongly affected by species composition.

### Practice 3 — Hunt for Another Reversal (harder)
Loop over **every pair** of measurements. For each pair, compute the combined correlation and the three within-species correlations. Flag any pair where the combined sign differs from **all three** species' signs. Is bill length/depth the only one?

In [21]:
# Your code here

from itertools import combinations
pair_rows = []
for x, y in combinations(MEASUREMENTS, 2):
    overall_r = clean[x].corr(clean[y])
    row = {'x': x, 'y': y, 'overall': overall_r}
    group_rs = []
    for species, group in clean.groupby('species', observed=True):
        value = group[x].corr(group[y])
        row[str(species)] = value
        group_rs.append(value)
    row['sign_reversal_all_species'] = bool(
        np.isfinite(overall_r) and all(np.isfinite(value) and
        np.sign(value) != 0 and np.sign(overall_r) != 0 and
        np.sign(value) != np.sign(overall_r) for value in group_rs))
    pair_rows.append(row)
pair_report = pd.DataFrame(pair_rows)
print(pair_report.round(6).to_string(index=False))
print('Pairs reversed:', pair_report.loc[pair_report['sign_reversal_all_species'], ['x', 'y']].to_dict('records'))


                x                 y   overall   Adelie  Chinstrap   Gentoo  sign_reversal_all_species
   bill_length_mm     bill_depth_mm -0.235053 0.391492   0.653536 0.643384                       True
   bill_length_mm flipper_length_mm  0.656181 0.325785   0.471607 0.661162                      False
   bill_length_mm       body_mass_g  0.595110 0.548866   0.513638 0.669166                      False
    bill_depth_mm flipper_length_mm -0.583851 0.307620   0.580143 0.706563                       True
    bill_depth_mm       body_mass_g -0.471916 0.576138   0.604498 0.719085                       True
flipper_length_mm       body_mass_g  0.871202 0.468202   0.641559 0.702667                      False
Pairs reversed: [{'x': 'bill_length_mm', 'y': 'bill_depth_mm'}, {'x': 'bill_depth_mm', 'y': 'flipper_length_mm'}, {'x': 'bill_depth_mm', 'y': 'body_mass_g'}]


**Practice 3 response:** Bill length/depth is not the only reversal. Three of six pairs reverse from negative overall to positive within **every** species: bill length/depth (overall −0.235), bill depth/flipper length (−0.584), and bill depth/body mass (−0.472). The complete table reports all six pooled correlations and all eighteen within-species correlations, rather than selecting just the reversed cases.

## ✏️ Challenge Task
Build a **sex comparison report**: for each species, the number of males and females, and the male − female difference in mean `body_mass_g`, `flipper_length_mm`, and `bill_depth_mm`. Save it as `OUT / 'sex_differences_by_species.csv'`.

In [22]:
# Your code here

# Unknown sex remains in clean, but does not enter male/female comparisons.
known_sex = clean.loc[clean['sex'].isin(['Male', 'Female'])]
sex_n = known_sex.groupby(['species', 'sex'], observed=True).size().unstack('sex')
sex_means = known_sex.groupby(['species', 'sex'], observed=True)[
    ['body_mass_g', 'flipper_length_mm', 'bill_depth_mm']].mean()
sex_report = pd.DataFrame({'male_n': sex_n['Male'], 'female_n': sex_n['Female']})
for measurement in ['body_mass_g', 'flipper_length_mm', 'bill_depth_mm']:
    wide = sex_means[measurement].unstack('sex')
    sex_report[measurement + '_male_minus_female'] = wide['Male'] - wide['Female']
sex_report.to_csv(OUT / 'sex_differences_by_species.csv')
print(sex_report.round(6).to_string())
print('Known-sex rows:', len(known_sex), '| unknown-sex rows retained in clean:', clean['sex'].isna().sum())


           male_n  female_n  body_mass_g_male_minus_female  flipper_length_mm_male_minus_female  bill_depth_mm_male_minus_female
species                                                                                                                         
Adelie         73        73                     674.657534                             4.616438                         1.450685
Chinstrap      34        34                     411.764706                             8.176471                         1.664706
Gentoo         61        58                     805.094686                             8.834087                         1.480102
Known-sex rows: 333 | unknown-sex rows retained in clean: 9


**Sex report interpretation:** All three male-minus-female mean differences are positive in every species. Counts are 73 male and 73 female Adelies, 34 and 34 Chinstraps, and 61 and 58 Gentoos (333 known-sex birds total). These are descriptive comparisons within species; the nine unknown-sex birds remain in the cleaned dataset and are excluded only from this report. Differences are calculated from unrounded group means and saved with their measurement units in column names.

## 🔥 Challenge (Optional) — Same Workflow, New Data
Run sections 1–7 on `pokemon.csv` in a compact form: data dictionary, measurement summary with skew, outliers overall and within `Legendary`, a group comparison by `Generation`, and the strongest correlations. Save at least one table to `OUT` with a `pokemon_` prefix. Write three findings in a Markdown cell.

In [23]:
# Your code here

*Your three findings here.*

**Optional challenge status:** The new-data Pokémon challenge is queued until course-wide required work is complete. No optional findings are asserted.

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `06-OutputFiles`. Upload that folder along with this notebook.

In [24]:
# Independent numeric audit: use NumPy on the raw measured rows.
raw_measured = penguins.loc[penguins[MEASUREMENTS].notna().all(axis=1)].copy()
assert len(penguins) == 344 and len(clean) == 342
assert penguins[MEASUREMENTS].isna().any(axis=1).equals(penguins[MEASUREMENTS].isna().all(axis=1))
assert clean['sex'].isna().sum() == 9 and len(known_sex) == 333
assert (clean[MEASUREMENTS].to_numpy() > 0).all()
np.testing.assert_array_equal(raw_measured[MEASUREMENTS].to_numpy(), clean[MEASUREMENTS].to_numpy())
np.testing.assert_allclose(np.mean(raw_measured[MEASUREMENTS], axis=0), overall['mean'])
np.testing.assert_allclose(np.std(raw_measured[MEASUREMENTS], axis=0, ddof=1), overall['std'])
np.testing.assert_allclose(np.corrcoef(raw_measured[MEASUREMENTS].to_numpy().T), corr.to_numpy())

fence_rows = []
for species_name, frame in [('ALL', raw_measured)] + list(raw_measured.groupby('species')):
    any_flags = np.zeros(len(frame), dtype=bool)
    for measurement in MEASUREMENTS:
        values = frame[measurement].to_numpy()
        q1_np, q3_np = np.quantile(values, [.25, .75], method='linear')
        lower_np = q1_np - 1.5 * (q3_np - q1_np)
        upper_np = q3_np + 1.5 * (q3_np - q1_np)
        flags_np = (values < lower_np) | (values > upper_np)
        any_flags |= flags_np
        np.testing.assert_allclose(iqr_fences(frame[measurement]), [lower_np, upper_np])
        if species_name == 'ALL':
            assert not flags_np.any()
        else:
            np.testing.assert_array_equal(within.loc[frame.index, measurement], flags_np)
        fence_rows.append({'group': species_name, 'measurement': measurement,
                           'lower': lower_np, 'upper': upper_np, 'flags': int(flags_np.sum())})
    if species_name != 'ALL':
        assert int(species_summary.loc[species_name, 'penguins']) == len(frame)
        for column, measurement, statistic in [
            ('mass_mean', 'body_mass_g', np.mean),
            ('mass_median', 'body_mass_g', np.median),
            ('flipper_mean', 'flipper_length_mm', np.mean),
            ('bill_length_mean', 'bill_length_mm', np.mean),
            ('bill_depth_mean', 'bill_depth_mm', np.mean)]:
            assert np.isclose(species_summary.loc[species_name, column],
                              round(float(statistic(frame[measurement])), 1))
        assert np.isclose(species_summary.loc[species_name, 'mass_std'],
                          round(float(np.std(frame.body_mass_g, ddof=1)), 1))
print('Independent fence table (strict outside comparisons):')
print(pd.DataFrame(fence_rows).to_string(index=False))
assert within.sum().tolist() == [1, 1, 2, 2]
assert within.any(axis=1).sum() == 6

# Verify all 24 correlations, including all eighteen within-species values.
for _, row in pair_report.iterrows():
    np.testing.assert_allclose(row['overall'], np.corrcoef(raw_measured[row['x']], raw_measured[row['y']])[0, 1])
    for species_name, frame in raw_measured.groupby('species'):
        np.testing.assert_allclose(row[species_name], np.corrcoef(frame[row['x']], frame[row['y']])[0, 1])
assert pair_report['sign_reversal_all_species'].sum() == 3

# Audit every sex count and difference from original uppercase raw labels.
for species_name, frame in raw_measured.groupby('species'):
    male = frame.loc[frame.sex == 'MALE']
    female = frame.loc[frame.sex == 'FEMALE']
    assert sex_report.loc[species_name, 'male_n'] == len(male)
    assert sex_report.loc[species_name, 'female_n'] == len(female)
    for measurement in ['body_mass_g', 'flipper_length_mm', 'bill_depth_mm']:
        independent_gap = np.mean(male[measurement].to_numpy()) - np.mean(female[measurement].to_numpy())
        np.testing.assert_allclose(sex_report.loc[species_name, measurement + '_male_minus_female'], independent_gap)
        assert independent_gap > 0

required_csvs = {'cleaned_penguins.csv', 'penguins_data_dictionary.csv',
    'measurement_summary.csv', 'species_summary.csv', 'correlation_matrix.csv',
    'bill_correlation_by_species.csv', 'sex_differences_by_species.csv'}
assert {path.name for path in OUT.glob('*.csv')} == required_csvs
assert len(pd.read_csv(OUT / 'cleaned_penguins.csv')) == 342
pd.testing.assert_frame_equal(pd.read_csv(OUT / 'sex_differences_by_species.csv', index_col='species'),
                             sex_report, check_index_type=False, check_categorical=False)
print('PASS: seven required CSVs, measurements, counts, all fences, species statistics, all correlations, and sex differences.')


Independent fence table (strict outside comparisons):
    group       measurement     lower     upper  flags
      ALL    bill_length_mm   25.3125   62.4125      0
      ALL     bill_depth_mm   10.9500   23.3500      0
      ALL flipper_length_mm  155.5000  247.5000      0
      ALL       body_mass_g 1750.0000 6550.0000      0
   Adelie    bill_length_mm   30.7500   46.7500      0
   Adelie     bill_depth_mm   15.2500   21.2500      1
   Adelie flipper_length_mm  172.5000  208.5000      2
   Adelie       body_mass_g 2375.0000 4975.0000      0
Chinstrap    bill_length_mm   39.2625   58.1625      0
Chinstrap     bill_depth_mm   14.6500   22.2500      0
Chinstrap flipper_length_mm  176.0000  216.0000      0
Chinstrap       body_mass_g 2793.7500 4643.7500      2
   Gentoo    bill_length_mm   38.9250   55.9250      1
   Gentoo     bill_depth_mm   11.9500   17.9500      0
   Gentoo flipper_length_mm  198.5000  234.5000      0
   Gentoo       body_mass_g 3500.0000 6700.0000      0


PASS: seven required CSVs, measurements, counts, all fences, species statistics, all correlations, and sex differences.


In [25]:
for path in sorted(OUT.rglob('*')):
    print(path)

06-OutputFiles\bill_correlation_by_species.csv
06-OutputFiles\cleaned_penguins.csv
06-OutputFiles\correlation_matrix.csv
06-OutputFiles\measurement_summary.csv
06-OutputFiles\penguins_data_dictionary.csv
06-OutputFiles\sex_differences_by_species.csv
06-OutputFiles\species_summary.csv


---
### 📝 Summary
This lab described a dataset from scratch, with numbers only:
- **Inspected** first, removed only rows with nothing to contribute, and kept partial rows
- **Recorded meaning** with a data dictionary and `category` dtypes
- **Summarized** center, spread, and shape, then checked outliers overall **and within groups**
- **Compared species** fairly, and spotted island as a stand-in for species
- **Tested a relationship within groups** and found Simpson's paradox

🎓 That's Module 07. **Module 08** picks up right here: the same questions, answered with charts. Its first notebook starts with the Anscombe datasets from notebook 05, and its lab returns to these penguins.

### Numeric report: opening questions answered
The data describe individual penguins using nominal species/island/sex categories and four positive ratio-scale measurements; the exported dictionary records units, dtypes, and missingness. Typical values and spread are in `measurement_summary.csv`; body mass is most variable relative to its mean. There are no overall IQR outliers but six within-species flags, retained after inspection. Gentoo is heaviest and has longer flippers and shallower bills; species composition complicates island comparisons. Flipper length/body mass has the strongest pooled correlation (r ≈ 0.871), and all six measurement pairs are checked within species in Practice 3. Three pairs reverse sign after grouping, illustrating why pooled relationships alone can mislead. All seven required report CSVs are saved; the optional Pokémon extension remains queued.